<a href="https://colab.research.google.com/github/Chikka-Pradhayani/ABTalks-60-Days-AI-Challenge/blob/main/Day-10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re
import nltk
import numpy as np

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [3]:
class PreprocessingModule:

    def __init__(self):
        self.stop_words = set(stopwords.words("english"))
        self.lemmatizer = WordNetLemmatizer()

    def transform(self, text):
        # Check for empty input
        if not isinstance(text, str) or not text.strip():
            raise ValueError("Input text cannot be empty.")

        # Convert to lowercase
        text = text.lower()

        # Remove numbers and special characters
        text = re.sub(r'[^a-zA-Z\s]', ' ', text)

        # Remove extra spaces
        text = re.sub(r'\s+', ' ', text).strip()

        # Check if anything meaningful remains
        if not text:
            raise ValueError("Input contains no meaningful text.")

        # Tokenization
        tokens = text.split()

        # Remove stopwords
        tokens = [
            word for word in tokens
            if word not in self.stop_words
        ]

        # Lemmatization
        tokens = [
            self.lemmatizer.lemmatize(word)
            for word in tokens
        ]

        # Check if tokens remain
        if not tokens:
            raise ValueError("Input contains no meaningful words.")

        return " ".join(tokens)

In [4]:
preprocessor = PreprocessingModule()

text = "Machine Learning is AMAZING! It can analyze 1000+ data points."

result = preprocessor.transform(text)

print("Original:", text)
print("Processed:", result)

Original: Machine Learning is AMAZING! It can analyze 1000+ data points.
Processed: machine learning amazing analyze data point


In [5]:
class VectorizerModule:

    def __init__(self):
        self.vectorizer = TfidfVectorizer()
        self.corpus_vectors = None
        self.corpus = None

    def fit(self, corpus):
        self.corpus = corpus

        # Convert documents into TF-IDF vectors
        self.corpus_vectors = self.vectorizer.fit_transform(corpus)

    def transform(self, query):
        if self.corpus_vectors is None:
            raise ValueError("Vectorizer must be fitted before transforming a query.")

        # Convert query into TF-IDF vector
        query_vector = self.vectorizer.transform([query])

        # Calculate cosine similarity
        similarity_scores = cosine_similarity(
            query_vector,
            self.corpus_vectors
        )[0]

        return similarity_scores

In [6]:
corpus = [
    "Python is widely used for machine learning.",
    "Java is a popular programming language.",
    "Machine learning algorithms learn patterns from data.",
    "Artificial intelligence is changing modern technology.",
    "Cloud computing provides scalable computing resources."
]

vectorizer = VectorizerModule()

vectorizer.fit(corpus)

query = "machine learning"

scores = vectorizer.transform(query)

print("Similarity Scores:")
for i, score in enumerate(scores):
    print(f"Document {i + 1}: {score:.4f}")

Similarity Scores:
Document 1: 0.4758
Document 2: 0.0000
Document 3: 0.4545
Document 4: 0.0000
Document 5: 0.0000


In [7]:
class Pipeline:

    def __init__(self, preprocessor, vectorizer):
        self.preprocessor = preprocessor
        self.vectorizer = vectorizer

    def run(self, query, corpus):

        # Validate query
        if not isinstance(query, str) or not query.strip():
            raise ValueError("Query cannot be empty.")

        # Preprocess query
        processed_query = self.preprocessor.transform(query)

        # Preprocess all documents
        processed_corpus = [
            self.preprocessor.transform(document)
            for document in corpus
        ]

        # Fit vectorizer using processed corpus
        self.vectorizer.fit(processed_corpus)

        # Calculate similarity scores
        scores = self.vectorizer.transform(processed_query)

        # Create ranked results
        ranked_results = []

        for index, score in enumerate(scores):
            ranked_results.append({
                "document_id": index + 1,
                "document": corpus[index],
                "similarity_score": float(score)
            })

        # Sort from highest similarity to lowest
        ranked_results.sort(
            key=lambda x: x["similarity_score"],
            reverse=True
        )

        return ranked_results

In [11]:
preprocessor = PreprocessingModule()
vectorizer = VectorizerModule()

pipeline = Pipeline(
    preprocessor,
    vectorizer
)

print("Pipeline created successfully!")

Pipeline created successfully!


In [12]:
corpus = [
    "Python is widely used for machine learning.",
    "Java is a popular programming language.",
    "Machine learning algorithms learn patterns from data.",
    "Artificial intelligence is changing modern technology.",
    "Cloud computing provides scalable computing resources."
]

query = "machine learning"

results = pipeline.run(query, corpus)

for rank, result in enumerate(results, start=1):
    print(
        f"Rank {rank}: "
        f"Document {result['document_id']} | "
        f"Score: {result['similarity_score']:.4f}"
    )

Rank 1: Document 1 | Score: 0.5501
Rank 2: Document 3 | Score: 0.4955
Rank 3: Document 2 | Score: 0.0000
Rank 4: Document 4 | Score: 0.0000
Rank 5: Document 5 | Score: 0.0000


In [13]:
corpus = [
    "Python is a popular programming language used for software development and automation.",

    "Machine learning enables computers to learn patterns from data and make predictions.",

    "Artificial intelligence allows machines to perform tasks that normally require human intelligence.",

    "Java is widely used for object oriented programming and enterprise application development.",

    "Data science combines statistics, programming, and machine learning to extract useful insights from data.",

    "Cloud computing provides on demand access to computing power, storage, and software services.",

    "Database management systems are used to store, organize, and retrieve large amounts of information.",

    "Natural language processing helps computers understand and process human language.",

    "Deep learning uses neural networks with multiple layers to solve complex machine learning problems.",

    "Cybersecurity protects computer systems, networks, and data from unauthorized access and attacks.",

    "Web development involves creating websites and web applications using frontend and backend technologies.",

    "Computer networks allow devices to communicate and share information with each other.",

    "Software engineering focuses on designing, developing, testing, and maintaining reliable software.",

    "Big data technologies help organizations process and analyze extremely large and complex datasets.",

    "Generative AI can create text, images, code, and other content using trained artificial intelligence models."
]

In [14]:
print("Total documents:", len(corpus))

Total documents: 15


In [15]:
for i, document in enumerate(corpus, start=1):
    print(f"Document {i}: {document}")

Document 1: Python is a popular programming language used for software development and automation.
Document 2: Machine learning enables computers to learn patterns from data and make predictions.
Document 3: Artificial intelligence allows machines to perform tasks that normally require human intelligence.
Document 4: Java is widely used for object oriented programming and enterprise application development.
Document 5: Data science combines statistics, programming, and machine learning to extract useful insights from data.
Document 6: Cloud computing provides on demand access to computing power, storage, and software services.
Document 7: Database management systems are used to store, organize, and retrieve large amounts of information.
Document 8: Natural language processing helps computers understand and process human language.
Document 9: Deep learning uses neural networks with multiple layers to solve complex machine learning problems.
Document 10: Cybersecurity protects computer s

In [16]:
queries = [
    "machine learning algorithms",
    "Python programming",
    "cloud computing",
    "computer security",
    "artificial intelligence"
]

print("Total queries:", len(queries))

Total queries: 5


In [17]:
for query in queries:

    print("\n" + "=" * 70)
    print("QUERY:", query)
    print("=" * 70)

    results = pipeline.run(query, corpus)

    for rank, result in enumerate(results, start=1):
        print(
            f"Rank {rank}: "
            f"Document {result['document_id']} | "
            f"Similarity: {result['similarity_score']:.4f}"
        )


QUERY: machine learning algorithms
Rank 1: Document 9 | Similarity: 0.4830
Rank 2: Document 2 | Similarity: 0.3931
Rank 3: Document 5 | Similarity: 0.3364
Rank 4: Document 3 | Similarity: 0.1491
Rank 5: Document 1 | Similarity: 0.0000
Rank 6: Document 4 | Similarity: 0.0000
Rank 7: Document 6 | Similarity: 0.0000
Rank 8: Document 7 | Similarity: 0.0000
Rank 9: Document 8 | Similarity: 0.0000
Rank 10: Document 10 | Similarity: 0.0000
Rank 11: Document 11 | Similarity: 0.0000
Rank 12: Document 12 | Similarity: 0.0000
Rank 13: Document 13 | Similarity: 0.0000
Rank 14: Document 14 | Similarity: 0.0000
Rank 15: Document 15 | Similarity: 0.0000

QUERY: Python programming
Rank 1: Document 1 | Similarity: 0.5099
Rank 2: Document 4 | Similarity: 0.1727
Rank 3: Document 5 | Similarity: 0.1527
Rank 4: Document 2 | Similarity: 0.0000
Rank 5: Document 3 | Similarity: 0.0000
Rank 6: Document 6 | Similarity: 0.0000
Rank 7: Document 7 | Similarity: 0.0000
Rank 8: Document 8 | Similarity: 0.0000
Rank 

In [18]:
class Pipeline:

    def __init__(self, preprocessor, vectorizer):
        self.preprocessor = preprocessor
        self.vectorizer = vectorizer

    def run(self, query, corpus):

        # Edge case 1: Empty input
        if not isinstance(query, str) or not query.strip():
            raise ValueError("Query cannot be empty.")

        # Edge case 2: Single-character query
        if len(query.strip()) == 1:
            raise ValueError("Query must contain more than one character.")

        # Edge case 3: Numbers and symbols only
        if not re.search(r'[a-zA-Z]', query):
            raise ValueError(
                "Query must contain meaningful alphabetic characters."
            )

        # Preprocess query
        processed_query = self.preprocessor.transform(query)

        # Preprocess corpus
        processed_corpus = [
            self.preprocessor.transform(document)
            for document in corpus
        ]

        # Fit vectorizer
        self.vectorizer.fit(processed_corpus)

        # Calculate similarity
        scores = self.vectorizer.transform(processed_query)

        # Create ranked results
        ranked_results = []

        for index, score in enumerate(scores):
            ranked_results.append({
                "document_id": index + 1,
                "document": corpus[index],
                "similarity_score": float(score)
            })

        # Sort by similarity score
        ranked_results.sort(
            key=lambda x: x["similarity_score"],
            reverse=True
        )

        return ranked_results

In [19]:
edge_cases = [
    "",
    "a",
    "12345!!!"
]

for test_input in edge_cases:

    print("\nInput:", repr(test_input))

    try:
        results = pipeline.run(test_input, corpus)
        print("Result: Accepted")

    except ValueError as error:
        print("Result:", error)


Input: ''
Result: Query cannot be empty.

Input: 'a'
Result: Input contains no meaningful words.

Input: '12345!!!'
Result: Input contains no meaningful text.
